This notebook reads each Bronze table, casts the columns to real types, parses the dates, standardizes the text, and throws out rows that cannot be used. It also deduplicates DEMO so every case appears once, then trims the child tables to match.

This also shows the capability of folding a new quarter into tables that already exist using a Delta MERGE so that updated cases replace the older versions.  

In [ ]:
%pip install pyspark==3.5.0 delta-spark==3.2.0 -q

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = "/content/drive/MyDrive/faers-data-pipeline"
sys.path.insert(0, PROJECT_ROOT)

from src.utils.spark_session import get_spark
spark = get_spark("FAERS-Silver")

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

BRONZE_DIR = os.path.join(PROJECT_ROOT, "data/bronze")
SILVER_DIR = os.path.join(PROJECT_ROOT, "data/silver")

print(f"✓ Spark {spark.version} ready")
print(f"✓ Bronze: {BRONZE_DIR}")
print(f"✓ Silver: {SILVER_DIR}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
Mounted at /content/drive
✓ Spark 3.5.0 ready
✓ Bronze: /content/drive/MyDrive/faers-data-pipeline/data/bronze
✓ Silver: /content/drive/MyDrive/faers-data-pipeline/data/silver


Load all seven Bronze tables and print their row counts. These are the raw inputs, still every column a string, that the transforms below will clean.

In [ ]:
demo_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "demo"))
drug_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "drug"))
reac_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "reac"))
outc_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "outc"))
rpsr_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "rpsr"))
ther_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "ther"))
indi_bronze = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "indi"))

print(f"{'Table':<8} {'Bronze rows':>12}")
print("-" * 22)
for name, df in [("DEMO", demo_bronze), ("DRUG", drug_bronze),
                  ("REAC", reac_bronze), ("OUTC", outc_bronze),
                  ("RPSR", rpsr_bronze), ("THER", ther_bronze),
                  ("INDI", indi_bronze)]:
    print(f"{name:<8} {df.count():>12,}")

Table     Bronze rows
----------------------
DEMO          406,184
DRUG        1,909,327
REAC        1,445,416
OUTC          295,044
RPSR           12,381
THER          594,449
INDI        1,186,115


DEMO is the hardest table, with the dates arriving in three different lengths, ages coming in 6 different units, and cases appearing several times.  This function should fix all of this.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

def transform_demo_silver(demo_bronze):

    df = demo_bronze.drop("_source_file", "_ingestion_ts")

    # Filter out corrupt rows
    df = df.filter(F.col("primaryid").isNotNull() & (F.col("primaryid") != ""))

    # Type casting
    df = (df
        .withColumn("caseversion", F.col("caseversion").cast("int"))
        .withColumn("age", F.col("age").cast("double"))
        .withColumn("wt", F.col("wt").cast("double"))
    )

    # Parse event_dt (4, 6, or 8 characters)
    df = df.withColumn("event_dt_parsed",
        F.when(F.length("event_dt") == 8,
               F.to_date("event_dt", "yyyyMMdd"))
         .when(F.length("event_dt") == 6,
               F.to_date(F.concat("event_dt", F.lit("01")), "yyyyMMdd"))
         .when(F.length("event_dt") == 4,
               F.to_date(F.concat("event_dt", F.lit("0101")), "yyyyMMdd"))
         .otherwise(F.lit(None).cast("date"))
    )

    # Filter out impossible dates (before 1900 or in the future)
    df = df.withColumn("event_dt_parsed",
        F.when(
            (F.col("event_dt_parsed") >= F.lit("1900-01-01").cast("date")) &
            (F.col("event_dt_parsed") <= F.current_date()),
            F.col("event_dt_parsed")
        ).otherwise(F.lit(None).cast("date"))
    )

    # Normalize age to years
    df = df.withColumn("age_in_years",
        F.when(F.col("age_cod") == "YR", F.col("age"))
         .when(F.col("age_cod") == "DEC", F.col("age") * 10)
         .when(F.col("age_cod") == "MON", F.col("age") / 12)
         .when(F.col("age_cod") == "WK", F.col("age") / 52)
         .when(F.col("age_cod") == "DY", F.col("age") / 365.25)
         .when(F.col("age_cod") == "HR", F.col("age") / 8766)
         .otherwise(F.lit(None).cast("double"))
    )

    # Round to 1 decimal place and clamp unreasonable values
    df = df.withColumn("age_in_years",
        F.when((F.col("age_in_years") >= 0) & (F.col("age_in_years") <= 120),
               F.round("age_in_years", 1))
         .otherwise(F.lit(None).cast("double"))
    )

    # Standardize sex
    df = df.withColumn("sex",
        F.when(F.col("sex").isin("M", "F"), F.col("sex"))
         .when(F.col("sex") == "UNK", "UNK")
         .when(F.col("sex") == "NS", "NS")
         .otherwise("UNK")
    )

    # Trim whitespace on string columns
    for col_name in ["reporter_country", "occr_country", "mfr_sndr", "occp_cod"]:
        if col_name in df.columns:
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Deduplicate. Keep the latest caseversion per caseid.
    window = Window.partitionBy("caseid").orderBy(
        F.col("caseversion").desc(),
        F.col("fda_dt").desc(),
        F.col("primaryid").desc()
    )
    df = (df
        .withColumn("_rn", F.row_number().over(window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    return df


demo_silver = transform_demo_silver(demo_bronze)

before = demo_bronze.count()
after = demo_silver.count()
dupes_removed = before - after
print(f"DEMO: {before:,} Bronze → {after:,} Silver ({dupes_removed:,} duplicates removed)")

print("\nSample Silver DEMO:")
demo_silver.select(
    "primaryid", "caseid", "caseversion", "event_dt", "event_dt_parsed",
    "age", "age_cod", "age_in_years", "sex", "reporter_country"
).limit(10).toPandas()

DEMO: 406,184 Bronze → 406,184 Silver (0 duplicates removed)

Sample Silver DEMO:


,primaryid,caseid,caseversion,event_dt,event_dt_parsed,age,age_cod,age_in_years,sex,reporter_country
0,1002872124,10028721,24,20171025,2017-10-25,57.0,YR,57.0,F,CA
1,1005450710,10054507,10,None,None,68.0,YR,68.0,F,US
2,1005762118,10057621,18,20140204,2014-02-04,57.0,YR,57.0,M,CA
3,101539902,10153990,2,20140301,2014-03-01,44.0,YR,44.0,F,US
4,1017130215,10171302,15,None,None,63.0,YR,63.0,F,US
5,1020101729,10201017,29,20140428,2014-04-28,74.0,YR,74.0,M,CA
6,102081283,10208128,3,20100101,2010-01-01,38.0,YR,38.0,M,US
7,1023164241,10231642,41,20120505,2012-05-05,55.0,YR,55.0,F,CA
8,1023686419,10236864,19,20130801,2013-08-01,52.0,YR,52.0,F,US
9,102380074,10238007,4,20130501,2013-05-01,NaN,None,NaN,F,JP


The DRUG table is more simple, with the same drup showing up in different spellings and casing across different reports.  Everything is standardized here.

In [ ]:
def transform_drug_silver(drug_bronze):

    df = drug_bronze.drop("_source_file", "_ingestion_ts")

    # Filter corrupt rows
    df = df.filter(F.col("primaryid").isNotNull() & (F.col("primaryid") != ""))

    # Uppercase and trim drug names
    df = (df
        .withColumn("drugname", F.upper(F.trim(F.col("drugname"))))
        .withColumn("prod_ai", F.upper(F.trim(F.col("prod_ai"))))
    )

    # Cast types
    df = df.withColumn("drug_seq", F.col("drug_seq").cast("int"))

    # Standardize role_cod (trim whitespace, uppercase)
    df = df.withColumn("role_cod", F.upper(F.trim(F.col("role_cod"))))

    # Trim other string columns
    for col_name in ["route", "dose_form", "dose_freq"]:
        if col_name in df.columns:
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    return df


drug_silver = transform_drug_silver(drug_bronze)
print(f"DRUG: {drug_bronze.count():,} Bronze → {drug_silver.count():,} Silver")

# Verify the name standardization worked
print("\nDrug name standardization check (atorvastatin):")
drug_silver.filter(
    F.col("drugname").contains("ATORVA") | F.col("drugname").contains("LIPITOR")
).select("drugname", "prod_ai", "role_cod").distinct().show(10, truncate=False)

DRUG: 1,909,327 Bronze → 1,909,327 Silver

Drug name standardization check (atorvastatin):
+-----------------------------------------+-----------------------------------------+--------+
|drugname                                 |prod_ai                                  |role_cod|
+-----------------------------------------+-----------------------------------------+--------+
|ATORVASTATIN CALCIUM                     |ATORVASTATIN CALCIUM                     |C       |
|ATORVASTATIN CALCIUM                     |ATORVASTATIN CALCIUM                     |SS      |
|ATORVASTATIN CALCIUM\EZETIMIBE           |ATORVASTATIN CALCIUM\EZETIMIBE           |C       |
|AMLODIPINE\ATORVASTATIN\PERINDOPRIL      |AMLODIPINE\ATORVASTATIN\PERINDOPRIL      |C       |
|ATORVASTATIN CALCIUM\EZETIMIBE           |ATORVASTATIN CALCIUM\EZETIMIBE           |PS      |
|ATORVASTATIN                             |ATORVASTATIN                             |PS      |
|ATORVASTATIN CALCIUM TRIHYDRATE\EZETIMIBE|ATORVASTATI

REAC is simpler. Standardize the reaction term and drop any row that has no term at all, since a reaction you cannot name is not worth keeping.

In [ ]:
def transform_reac_silver(reac_bronze):
    """
    Clean the REAC table for the Silver layer.

    This drops the Bronze tracking columns and removes any row missing a
    primaryid or a reaction term, since a reaction you cannot name tells you
    nothing. Then it uppercases and trims the MedDRA preferred term.
    """

    df = reac_bronze.drop("_source_file", "_ingestion_ts")

    # Filter corrupt rows. A reaction without a term is useless.
    df = df.filter(
        F.col("primaryid").isNotNull() & (F.col("primaryid") != "") &
        F.col("pt").isNotNull() & (F.col("pt") != "")
    )

    # Standardize the MedDRA preferred term
    df = df.withColumn("pt", F.upper(F.trim(F.col("pt"))))

    return df


reac_silver = transform_reac_silver(reac_bronze)
print(f"REAC: {reac_bronze.count():,} Bronze → {reac_silver.count():,} Silver")

REAC: 1,445,416 Bronze → 1,445,416 Silver


The four smaller tables get the same basic treatment. Drop the tracking columns, remove rows with no primaryid, standardize their codes, and parse the therapy dates the same way DEMO parses event dates.

In [ ]:
def transform_outc_silver(outc_bronze):
    df = outc_bronze.drop("_source_file", "_ingestion_ts")
    df = df.filter(F.col("primaryid").isNotNull() & (F.col("primaryid") != ""))
    df = df.withColumn("outc_cod", F.upper(F.trim(F.col("outc_cod"))))
    return df


def transform_rpsr_silver(rpsr_bronze):
    df = rpsr_bronze.drop("_source_file", "_ingestion_ts")
    df = df.filter(F.col("primaryid").isNotNull() & (F.col("primaryid") != ""))
    df = df.withColumn("rpsr_cod", F.upper(F.trim(F.col("rpsr_cod"))))
    return df


def transform_ther_silver(ther_bronze):
    df = ther_bronze.drop("_source_file", "_ingestion_ts")
    df = df.filter(F.col("primaryid").isNotNull() & (F.col("primaryid") != ""))
    df = df.withColumn("dsg_drug_seq", F.col("dsg_drug_seq").cast("int"))

    # Parse therapy dates (same 4/6/8 char logic)
    for date_col in ["start_dt", "end_dt"]:
        if date_col in df.columns:
            df = df.withColumn(f"{date_col}_parsed",
                F.when(F.length(date_col) == 8,
                       F.to_date(date_col, "yyyyMMdd"))
                 .when(F.length(date_col) == 6,
                       F.to_date(F.concat(F.col(date_col), F.lit("01")), "yyyyMMdd"))
                 .when(F.length(date_col) == 4,
                       F.to_date(F.concat(F.col(date_col), F.lit("0101")), "yyyyMMdd"))
                 .otherwise(F.lit(None).cast("date"))
            )
    return df


def transform_indi_silver(indi_bronze):
    df = indi_bronze.drop("_source_file", "_ingestion_ts")
    df = df.filter(F.col("primaryid").isNotNull() & (F.col("primaryid") != ""))
    df = df.withColumn("indi_drug_seq", F.col("indi_drug_seq").cast("int"))
    df = df.withColumn("indi_pt", F.upper(F.trim(F.col("indi_pt"))))
    return df


outc_silver = transform_outc_silver(outc_bronze)
rpsr_silver = transform_rpsr_silver(rpsr_bronze)
ther_silver = transform_ther_silver(ther_bronze)
indi_silver = transform_indi_silver(indi_bronze)

print(f"OUTC: {outc_bronze.count():,} → {outc_silver.count():,}")
print(f"RPSR: {rpsr_bronze.count():,} → {rpsr_silver.count():,}")
print(f"THER: {ther_bronze.count():,} → {ther_silver.count():,}")
print(f"INDI: {indi_bronze.count():,} → {indi_silver.count():,}")

OUTC: 295,044 → 295,044
RPSR: 12,381 → 12,381
THER: 594,449 → 594,449
INDI: 1,186,115 → 1,186,115


Deduplicating DEMO dropped some reports with means that the child rows tied to them are leftover.  We can use an inner join to clean these out.

In [ ]:
# Get the set of surviving primaryids from DEMO Silver
valid_ids = demo_silver.select("primaryid")

before_counts = {
    "DRUG": drug_silver.count(),
    "REAC": reac_silver.count(),
    "OUTC": outc_silver.count(),
    "RPSR": rpsr_silver.count(),
    "THER": ther_silver.count(),
    "INDI": indi_silver.count(),
}

# Inner join to keep only rows matching valid primaryids
drug_silver = drug_silver.join(valid_ids, on="primaryid", how="inner")
reac_silver = reac_silver.join(valid_ids, on="primaryid", how="inner")
outc_silver = outc_silver.join(valid_ids, on="primaryid", how="inner")
rpsr_silver = rpsr_silver.join(valid_ids, on="primaryid", how="inner")
ther_silver = ther_silver.join(valid_ids, on="primaryid", how="inner")
indi_silver = indi_silver.join(valid_ids, on="primaryid", how="inner")

print("Orphan removal (rows dropped by filtering to valid primaryids):")
for name, df in [("DRUG", drug_silver), ("REAC", reac_silver),
                  ("OUTC", outc_silver), ("RPSR", rpsr_silver),
                  ("THER", ther_silver), ("INDI", indi_silver)]:
    after = df.count()
    dropped = before_counts[name] - after
    print(f"  {name}: {dropped:,} orphan rows removed → {after:,} remain")

Orphan removal (rows dropped by filtering to valid primaryids):
  DRUG: 0 orphan rows removed → 1,909,327 remain
  REAC: 0 orphan rows removed → 1,445,416 remain
  OUTC: 0 orphan rows removed → 295,044 remain
  RPSR: 0 orphan rows removed → 12,381 remain
  THER: 0 orphan rows removed → 594,449 remain
  INDI: 0 orphan rows removed → 1,186,115 remain


Write all seven cleaned tables to Delta in overwrite mode. The extra date setting lets Spark write very old dates without complaining.

In [ ]:
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.delta.datetimeRebaseModeInWrite", "CORRECTED")

silver_tables = {
    "demo": demo_silver,
    "drug": drug_silver,
    "reac": reac_silver,
    "outc": outc_silver,
    "rpsr": rpsr_silver,
    "ther": ther_silver,
    "indi": indi_silver,
}

print(f"Writing Silver tables to: {SILVER_DIR}\n")

for name, df in silver_tables.items():
    output_path = os.path.join(SILVER_DIR, name)
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(output_path))
    print(f"  ✓ {name}: {df.count():,} rows")

print(f"\n✓ All Silver tables written")

Writing Silver tables to: /content/drive/MyDrive/faers-data-pipeline/data/silver

  ✓ demo: 406,184 rows
  ✓ drug: 1,909,327 rows
  ✓ reac: 1,445,416 rows
  ✓ outc: 295,044 rows
  ✓ rpsr: 12,381 rows
  ✓ ther: 594,449 rows
  ✓ indi: 1,186,115 rows

✓ All Silver tables written


Read the tables back, confirm the row and column counts, check that DEMO now carries real types instead of strings, and prove that no child row points to a missing report.

In [ ]:
print(f"{'Table':<8} {'Rows':>10} {'Columns':>9}")
print("-" * 30)

for name in ["demo", "drug", "reac", "outc", "rpsr", "ther", "indi"]:
    delta_path = os.path.join(SILVER_DIR, name)
    df = spark.read.format("delta").load(delta_path)
    print(f"{name.upper():<8} {df.count():>10,} {len(df.columns):>9}")

# Verify DEMO types are correct (not all strings anymore)
print("\nDEMO Silver schema (key columns):")
demo_check = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo"))
for field in demo_check.schema.fields:
    if field.name in ["primaryid", "caseid", "caseversion", "event_dt",
                       "event_dt_parsed", "age", "age_in_years", "sex",
                       "reporter_country", "_year", "_quarter"]:
        print(f"  {field.name:<20} {str(field.dataType)}")

# Verify no orphans. Every child primaryid exists in DEMO.
print("\nReferential integrity check:")
demo_ids = demo_check.select("primaryid").distinct()
for name in ["drug", "reac", "outc"]:
    child = spark.read.format("delta").load(os.path.join(SILVER_DIR, name))
    orphans = child.select("primaryid").distinct().subtract(demo_ids).count()
    status = "✓ PASS" if orphans == 0 else f"✗ FAIL ({orphans} orphans)"
    print(f"  {name.upper()} → DEMO: {status}")

Table          Rows   Columns
------------------------------
DEMO        406,184        29
DRUG      1,909,327        22
REAC      1,445,416         6
OUTC        295,044         5
RPSR         12,381         5
THER        594,449        11
INDI      1,186,115         6

DEMO Silver schema (key columns):
  primaryid            StringType()
  caseid               StringType()
  caseversion          IntegerType()
  event_dt             StringType()
  age                  DoubleType()
  sex                  StringType()
  reporter_country     StringType()
  _year                StringType()
  _quarter             StringType()
  event_dt_parsed      DateType()
  age_in_years         DoubleType()

Referential integrity check:
  DRUG → DEMO: ✓ PASS
  REAC → DEMO: ✓ PASS
  OUTC → DEMO: ✓ PASS


Now for the checks that the transforms worked.  

In [ ]:
demo_s = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo"))
drug_s = spark.read.format("delta").load(os.path.join(SILVER_DIR, "drug"))

# Show the 3 different input lengths.
print("Date parsing results (event_dt → event_dt_parsed):")
demo_s.select("event_dt", "event_dt_parsed") \
    .filter(F.col("event_dt").isNotNull()) \
    .dropDuplicates(["event_dt"]) \
    .orderBy(F.length("event_dt")) \
    .limit(9).toPandas()
# 2. Show conversions from different units.
print("\nAge normalization results:")
demo_s.select("age", "age_cod", "age_in_years") \
    .filter(F.col("age").isNotNull() & F.col("age_cod").isNotNull()) \
    .dropDuplicates(["age_cod"]) \
    .orderBy("age_cod") \
    .toPandas()
# 3. Confirm all uppercase with no leading or trailing spaces.
print("\nDrug name check (should all be uppercase, trimmed):")
lowercase_drugs = drug_s.filter(F.col("drugname") != F.upper(F.col("drugname")))
print(f"  Drugs still with lowercase: {lowercase_drugs.count()} (should be 0)")

space_drugs = drug_s.filter(F.col("drugname") != F.trim(F.col("drugname")))
print(f"  Drugs with leading/trailing spaces: {space_drugs.count()} (should be 0)")

# 4. Confirm no duplicate caseids in DEMO Silver.
from pyspark.sql.functions import countDistinct
total_rows = demo_s.count()
unique_cases = demo_s.select("caseid").distinct().count()
print(f"\n  DEMO rows: {total_rows:,}")
print(f"  Unique caseids: {unique_cases:,}")
print(f"  Match: {'✓ YES' if total_rows == unique_cases else '✗ NO — still has duplicates!'}")

Date parsing results (event_dt → event_dt_parsed):

Age normalization results:

Drug name check (should all be uppercase, trimmed):
  Drugs still with lowercase: 0 (should be 0)
  Drugs with leading/trailing spaces: 0 (should be 0)

  DEMO rows: 406,184
  Unique caseids: 406,184
  Match: ✓ YES


Now the pipeline is also stored in silver_transforms.py

In [ ]:
from src.ingestion.bronze_ingestion import ingest_quarter_to_bronze

results = ingest_quarter_to_bronze(spark, PROJECT_ROOT, "2024", "2")

Ingesting 2024 Q2
  done: DEMO: 397,119 rows from DEMO24Q2.txt
  done: DRUG: 1,888,937 rows from DRUG24Q2.txt
  done: REAC: 1,445,044 rows from REAC24Q2.txt
  done: OUTC: 291,572 rows from OUTC24Q2.txt
  done: RPSR: 11,517 rows from RPSR24Q2.txt
  done: THER: 539,334 rows from THER24Q2.txt
  done: INDI: 1,187,626 rows from INDI24Q2.txt
  Total: 5,761,149 rows for 2024 Q2


Now it goes through the Silver Pipeline


In [ ]:
from src.transformations.silver_transforms import (
    transform_demo_silver, transform_drug_silver, transform_reac_silver,
    transform_outc_silver, transform_rpsr_silver, transform_ther_silver,
    transform_indi_silver, filter_to_valid_ids
)

# Read Q2 from Bronze (filter to only Q2 data)
demo_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "demo")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")
drug_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "drug")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")
reac_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "reac")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")
outc_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "outc")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")
rpsr_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "rpsr")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")
ther_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "ther")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")
indi_bronze_q2 = spark.read.format("delta").load(os.path.join(BRONZE_DIR, "indi")) \
    .filter("_year = '2024' AND _quarter = 'Q2'")

# Apply Silver transforms
demo_q2 = transform_demo_silver(demo_bronze_q2)
drug_q2 = transform_drug_silver(drug_bronze_q2)
reac_q2 = transform_reac_silver(reac_bronze_q2)
outc_q2 = transform_outc_silver(outc_bronze_q2)
rpsr_q2 = transform_rpsr_silver(rpsr_bronze_q2)
ther_q2 = transform_ther_silver(ther_bronze_q2)
indi_q2 = transform_indi_silver(indi_bronze_q2)

# Filter child tables to valid DEMO primaryids
drug_q2 = filter_to_valid_ids(demo_q2, drug_q2)
reac_q2 = filter_to_valid_ids(demo_q2, reac_q2)
outc_q2 = filter_to_valid_ids(demo_q2, outc_q2)
rpsr_q2 = filter_to_valid_ids(demo_q2, rpsr_q2)
ther_q2 = filter_to_valid_ids(demo_q2, ther_q2)
indi_q2 = filter_to_valid_ids(demo_q2, indi_q2)

print("Q2 Silver transforms complete:")
for name, df in [("DEMO", demo_q2), ("DRUG", drug_q2), ("REAC", reac_q2),
                  ("OUTC", outc_q2), ("RPSR", rpsr_q2), ("THER", ther_q2),
                  ("INDI", indi_q2)]:
    print(f"  {name}: {df.count():,} rows")

Q2 Silver transforms complete:
  DEMO: 397,119 rows
  DRUG: 1,888,937 rows
  REAC: 1,445,044 rows
  OUTC: 291,572 rows
  RPSR: 11,517 rows
  THER: 539,334 rows
  INDI: 1,187,626 rows


Before merging we need to see how much the two quarters share.  Some cases show up in both quarters and those are the ones that need to be fixed.  The rest are just simply added over.


In [ ]:
demo_existing = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo"))

existing_cases = set(
    row.caseid for row in demo_existing.select("caseid").distinct().collect()
)
new_cases = set(
    row.caseid for row in demo_q2.select("caseid").distinct().collect()
)

overlap = existing_cases & new_cases
only_q1 = existing_cases - new_cases
only_q2 = new_cases - existing_cases

print(f"Existing Silver (Q1): {len(existing_cases):,} unique cases")
print(f"New data (Q2):        {len(new_cases):,} unique cases")
print(f"")
print(f"Overlap (same caseid in both):  {len(overlap):,}")
print(f"  → These will be UPDATED if Q2 has a newer caseversion")
print(f"Only in Q1 (untouched):         {len(only_q1):,}")
print(f"Only in Q2 (new inserts):       {len(only_q2):,}")

Existing Silver (Q1): 406,184 unique cases
New data (Q2):        397,119 unique cases

Overlap (same caseid in both):  35,529
  → These will be UPDATED if Q2 has a newer caseversion
Only in Q1 (untouched):         370,655
Only in Q2 (new inserts):       361,590



The two functions will do the merging.  One updates DEMO and the other refreshes a child table by deleting the rows for any case that has changed and appending the new ones.  


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import Window


def merge_demo_silver(spark, new_demo_df, silver_path):
    delta_path = os.path.join(silver_path, "demo")

    # Pre-deduplicate the source first.
    window = Window.partitionBy("caseid").orderBy(
        F.col("caseversion").desc(),
        F.col("fda_dt").desc(),
        F.col("primaryid").desc()
    )
    source = (new_demo_df
        .withColumn("_rn", F.row_number().over(window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    source_count = source.count()

    if not DeltaTable.isDeltaTable(spark, delta_path):
        source.write.format("delta").mode("overwrite").save(delta_path)
        return {"inserted": source_count, "updated": 0, "source_rows": source_count}

    delta_table = DeltaTable.forPath(spark, delta_path)
    before_count = spark.read.format("delta").load(delta_path).count()

    delta_table.alias("target").merge(
        source.alias("source"),
        "target.caseid = source.caseid"
    ).whenMatchedUpdateAll(
        condition="source.caseversion > target.caseversion"
    ).whenNotMatchedInsertAll(
    ).execute()

    after_count = spark.read.format("delta").load(delta_path).count()
    inserted = after_count - before_count
    updated = source_count - inserted

    return {"inserted": inserted, "updated": updated, "source_rows": source_count}


def merge_child_table(spark, new_df, silver_path, table_name, demo_silver_path):
    delta_path = os.path.join(silver_path, table_name)

    valid_ids = spark.read.format("delta").load(demo_silver_path) \
        .select("primaryid")

    new_valid = new_df.join(valid_ids, on="primaryid", how="inner")

    if not DeltaTable.isDeltaTable(spark, delta_path):
        new_valid.write.format("delta").mode("overwrite").save(delta_path)
        return {"written": new_valid.count()}

    new_primaryids = new_valid.select("primaryid").distinct()
    delta_table = DeltaTable.forPath(spark, delta_path)

    new_id_list = [row.primaryid for row in new_primaryids.collect()]

    if new_id_list:
        batch_size = 10000
        for i in range(0, len(new_id_list), batch_size):
            batch = new_id_list[i:i + batch_size]
            delta_table.delete(F.col("primaryid").isin(batch))

    new_valid.write.format("delta").mode("append").save(delta_path)

    return {"written": new_valid.count()}


print("✓ merge_demo_silver() and merge_child_table() defined")

✓ merge_demo_silver() and merge_child_table() defined



Here we run the DEMO merge and watch the counts. Inserted rows are brand new cases. Updated rows are cases that came back with a newer version.

In [ ]:
print("DEMO MERGE:")
print(f"  Silver before: {spark.read.format('delta').load(os.path.join(SILVER_DIR, 'demo')).count():,}")

demo_stats = merge_demo_silver(spark, demo_q2, SILVER_DIR)

print(f"  Source rows:   {demo_stats['source_rows']:,}")
print(f"  Inserted:      {demo_stats['inserted']:,}")
print(f"  Updated:       {demo_stats['updated']:,}")
print(f"  Silver after:  {spark.read.format('delta').load(os.path.join(SILVER_DIR, 'demo')).count():,}")

DEMO MERGE:
  Silver before: 406,184
  Source rows:   397,119
  Inserted:      361,590
  Updated:       35,529
  Silver after:  767,774


Apply the same merge to each child table. The before and after counts show how each one grew once Q2 was folded in.

In [ ]:
demo_silver_path = os.path.join(SILVER_DIR, "demo")

child_tables = {
    "drug": drug_q2,
    "reac": reac_q2,
    "outc": outc_q2,
    "rpsr": rpsr_q2,
    "ther": ther_q2,
    "indi": indi_q2,
}

print(f"{'Table':<8} {'Before':>10} {'Written':>10} {'After':>10}")
print("-" * 42)

for name, df in child_tables.items():
    delta_path = os.path.join(SILVER_DIR, name)
    before = spark.read.format("delta").load(delta_path).count()

    stats = merge_child_table(spark, df, SILVER_DIR, name, demo_silver_path)

    after = spark.read.format("delta").load(delta_path).count()
    print(f"{name.upper():<8} {before:>10,} {stats['written']:>10,} {after:>10,}")

Table        Before    Written      After
------------------------------------------
DRUG      1,909,327  1,888,937  3,798,262
REAC      1,445,416  1,445,044  2,890,456
OUTC        295,044    291,572    586,615
RPSR         12,381     11,517     23,897
THER        594,449    539,334  1,133,782
INDI      1,186,115  1,187,626  2,373,740


Now we need to make sure that children rows aren't pointing at cases that we removed.  


In [ ]:
demo_ids = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo")) \
    .select("primaryid").distinct()

print("Cleaning orphan child rows...\n")

for name in ["drug", "reac", "outc", "rpsr", "ther", "indi"]:
    delta_path = os.path.join(SILVER_DIR, name)
    child_df = spark.read.format("delta").load(delta_path)

    before = child_df.count()
    # Keep only rows whose primaryid exists in DEMO
    clean_df = child_df.join(demo_ids, on="primaryid", how="inner")
    after = clean_df.count()
    removed = before - after

    # Overwrite with cleaned data
    (clean_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(delta_path))

    print(f"  {name.upper()}: removed {removed:,} orphans → {after:,} rows")

print("\n✓ Orphan cleanup complete")

Cleaning orphan child rows...

  DRUG: removed 0 orphans → 3,519,718 rows
  REAC: removed 0 orphans → 2,683,365 rows
  OUTC: removed 0 orphans → 557,664 rows
  RPSR: removed 0 orphans → 23,897 rows
  THER: removed 0 orphans → 1,050,991 rows
  INDI: removed 0 orphans → 2,239,116 rows

✓ Orphan cleanup complete


Now we can see how the Delta records the merge as its own entry in the table history.

In [ ]:
from delta.tables import DeltaTable

demo_path = os.path.join(SILVER_DIR, "demo")
delta_table = DeltaTable.forPath(spark, demo_path)

print("DEMO Silver — Delta history:")
history = delta_table.history()
history.select(
    "version", "timestamp", "operation",
    "operationMetrics.numTargetRowsInserted",
    "operationMetrics.numTargetRowsUpdated",
    "operationMetrics.numOutputRows",
).show(truncate=False)

DEMO Silver — Delta history:
+-------+-------------------+---------+---------------------+--------------------+-------------+
|version|timestamp          |operation|numTargetRowsInserted|numTargetRowsUpdated|numOutputRows|
+-------+-------------------+---------+---------------------+--------------------+-------------+
|1      |2026-03-30 00:39:28|MERGE    |361590               |35528               |767774       |
|0      |2026-03-29 23:32:03|WRITE    |NULL                 |NULL                |406184       |
+-------+-------------------+---------+---------------------+--------------------+-------------+




Now we are proving once again that there are not duplicate cases in DEMO and no rows with no parent in the child tables.


In [ ]:
demo_silver = spark.read.format("delta").load(os.path.join(SILVER_DIR, "demo"))

total_rows = demo_silver.count()
unique_cases = demo_silver.select("caseid").distinct().count()

print(f"DEMO Silver after MERGE:")
print(f"  Total rows:    {total_rows:,}")
print(f"  Unique cases:  {unique_cases:,}")
print(f"  Duplicates:    {total_rows - unique_cases:,}")
print(f"  Status:        {'✓ NO DUPLICATES' if total_rows == unique_cases else '✗ HAS DUPLICATES'}")

# Verify referential integrity
demo_ids = demo_silver.select("primaryid").distinct()

print(f"\nReferential integrity after MERGE:")
for name in ["drug", "reac", "outc", "rpsr", "ther", "indi"]:
    child = spark.read.format("delta").load(os.path.join(SILVER_DIR, name))
    orphans = child.select("primaryid").distinct().subtract(demo_ids).count()
    status = "✓ PASS" if orphans == 0 else f"✗ FAIL ({orphans} orphans)"
    print(f"  {name.upper()} → DEMO: {status}")

DEMO Silver after MERGE:
  Total rows:    767,774
  Unique cases:  767,774
  Duplicates:    0
  Status:        ✓ NO DUPLICATES

Referential integrity after MERGE:
  DRUG → DEMO: ✓ PASS
  REAC → DEMO: ✓ PASS
  OUTC → DEMO: ✓ PASS
  RPSR → DEMO: ✓ PASS
  THER → DEMO: ✓ PASS
  INDI → DEMO: ✓ PASS


Now here is the table before the merge versus after the merge with the Q2 cases added.

In [ ]:
demo_path = os.path.join(SILVER_DIR, "demo")

history = DeltaTable.forPath(spark, demo_path).history()
versions = history.select("version", "operation").orderBy("version").collect()

print("Available versions:")
for v in versions:
    print(f"  Version {v.version}: {v.operation}")

first_version = versions[0].version
latest_version = versions[-1].version

df_before = spark.read.format("delta") \
    .option("versionAsOf", first_version).load(demo_path)
df_after = spark.read.format("delta") \
    .option("versionAsOf", latest_version).load(demo_path)

print(f"\nVersion {first_version} (Q1 only):      {df_before.count():,} cases")
print(f"Version {latest_version} (Q1+Q2 merged):  {df_after.count():,} cases")
print(f"Net new cases added:       {df_after.count() - df_before.count():,}")

Available versions:
  Version 0: WRITE
  Version 1: MERGE

Version 0 (Q1 only):      406,184 cases
Version 1 (Q1+Q2 merged):  767,774 cases
Net new cases added:       361,590
